# Scenario 3 — Multi-Agent System (Planner -> Executor -> Reviewer)

## Why Multi-Agent?

In Scenario 2, a single agent did everything: analyzed, searched, registered, and responded.
When complexity grows, it makes sense to **separate responsibilities**:

| Agent | Role |
|---|---|
| **Planner** | Understands the request, classifies it, and creates an execution plan |
| **Executor** | Executes the plan using the available tools |
| **Reviewer** | Reviews the results, identifies gaps, and generates a structured report |

## When does this make sense?

Imagine a complex request:
> *"I'm Maria Silva. My TechBook Essentials has a flickering screen. I opened a ticket before but the problem persists. I want to know the exchange policy, the status of my previous ticket, if this model has a manufacturer recall, and I want to open a new urgent ticket."*

This request requires:
1. **Planning** — decompose into 4 subtasks
2. **Execution** — query FAQ, CSV, web, and register a ticket
3. **Review** — consolidate everything and generate a clear report

## Architecture

```
[START] -> [PLANNER] -> [EXECUTOR] -> [REVIEWER] -> [END]
              |              |              |
          Decomposes     Executes each   Reviews, refines
          the request    subtask with    and generates a
          into tasks     the tools       Markdown report
```

## LangGraph Concepts

| Concept | Application |
|---|---|
| **Agents as nodes** | Each agent is a node in the graph |
| **Shared state** | Plan, results, and report flow between agents |
| **Specialization** | Each node has a clear responsibility |
| **Structured output** | The Reviewer generates exportable Markdown |

## Step 1 — Imports and Tools

In [ ]:
import os, json
from datetime import date, datetime
from dotenv import load_dotenv

load_dotenv()

import pandas as pd
from shared.llm import get_llm
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_databricks import DatabricksEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

llm = get_llm()

# -- Vector stores --
def create_store(file_path):
    loader = TextLoader(file_path, encoding="utf-8")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,
                                              separators=["\n## ", "\n### ", "\n\n", "\n", " "])
    embeddings = DatabricksEmbeddings(endpoint=os.getenv("DATABRICKS_EMBEDDINGS_ENDPOINT", "databricks-bge-large-en"))
    return FAISS.from_documents(splitter.split_documents(loader.load()), embeddings)

policies_store = create_store("data/kb/store_policies.md")
products_store = create_store("data/kb/product_faq.md")

# -- CSV --
CSV_PATH = "data/csv/tickets.csv"

def query_csv(filter_text: str) -> str:
    df = pd.read_csv(CSV_PATH)
    mask = df.apply(lambda row: filter_text.lower() in str(row).lower(), axis=1)
    res = df[mask]
    return res.to_string(index=False) if not res.empty else "No tickets found."

def register_ticket(customer, email, category, description, product="", priority="Medium") -> str:
    df = pd.read_csv(CSV_PATH)
    new_id = int(df["id"].max()) + 1
    new_row = {"id": new_id, "customer": customer, "email": email,
               "open_date": date.today().isoformat(), "category": category,
               "description": description, "status": "Open", "priority": priority,
               "product": product, "resolution": ""}
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return f"Ticket #{new_id} registered successfully!"

# -- FAQ search --
def search_faq(query: str) -> str:
    docs_pol = policies_store.similarity_search(query, k=2)
    docs_prod = products_store.similarity_search(query, k=2)
    return "\n\n".join(d.page_content for d in docs_pol + docs_prod)

# -- Web search --
_web_search = DuckDuckGoSearchRun()

def search_web(query: str) -> str:
    try:
        return _web_search.invoke(query)
    except Exception as e:
        return f"Web search error: {e}"

print("All tools ready: RAG, CSV, Web Search")

## Step 2 — Shared State Between Agents

The state now carries the **plan**, the **results**, and the **final report**.

```
plan                 -> created by the Planner
execution_results    -> filled by the Executor (accumulates)
final_response       -> generated by the Reviewer
report_md            -> exportable Markdown report
```

In [ ]:
class MultiAgentState(TypedDict):
    """State shared between the 3 agents."""
    question: str
    plan: dict                                     # Planner -> execution plan
    execution_results: Annotated[list, add]         # Executor -> results (accumulates)
    final_response: str                             # Reviewer -> response to customer
    report_md: str                                  # Reviewer -> Markdown report

print("State defined — MultiAgentState")

## Step 3 — Agent 1: Planner

The Planner receives the request and:
1. **Classifies** the type of request
2. **Decomposes** it into subtasks (query decomposition)
3. **Defines** which tools each task needs
4. Returns a **structured plan** in JSON

In [ ]:
def agent_planner(state: MultiAgentState) -> dict:
    """Planner Agent — decomposes the request into tasks."""

    question = state["question"]
    print("\n" + "=" * 70)
    print("PLANNER — Analyzing and planning...")
    print("=" * 70)
    print(f"   Request: '{question}'")

    prompt = f"""You are the Planner Agent for a TechMart (electronics store) customer support center.

Your task: analyze the customer's request and create an execution plan.

Available tools:
- "query_faq": search store policies and product FAQ
- "query_csv": search existing ticket records
- "search_web": search for information on the internet
- "register_ticket": create a new support ticket

Return ONLY a valid JSON (no markdown) with:
{{"request_type": "short description", "complexity": "simple or complex", "tasks": [{{"id": 1, "action": "tool_name", "description": "what to do", "parameters": {{"search": "term"}}}}]}}

Customer request: {question}"""

    resp = llm.invoke(prompt)
    try:
        plan = json.loads(resp.content)
    except json.JSONDecodeError:
        plan = {
            "request_type": "general query",
            "complexity": "simple",
            "tasks": [{"id": 1, "action": "query_faq",
                       "description": question, "parameters": {"search": question}}]
        }

    print(f"\n   Plan created:")
    print(f"   Type: {plan.get('request_type')}")
    print(f"   Complexity: {plan.get('complexity')}")
    print(f"   Tasks: {len(plan.get('tasks', []))}")
    for t in plan.get("tasks", []):
        print(f"     #{t['id']} [{t['action']}] — {t['description']}")

    return {"plan": plan}


print("Planner Agent defined")

## Step 4 — Agent 2: Executor

The Executor receives the plan and:
1. **Iterates** over each task in the plan
2. **Calls** the corresponding tool
3. **Collects** the results
4. Returns all accumulated results

In [ ]:
def agent_executor(state: MultiAgentState) -> dict:
    """Executor Agent — executes each task from the plan."""

    plan = state.get("plan", {})
    tasks = plan.get("tasks", [])

    print("\n" + "=" * 70)
    print(f"EXECUTOR — Executing {len(tasks)} task(s)...")
    print("=" * 70)

    results = []

    for task in tasks:
        tid = task.get("id", "?")
        action = task.get("action", "")
        desc = task.get("description", "")
        params = task.get("parameters", {})
        search_term = params.get("search", desc)

        print(f"\n   Task #{tid}: [{action}] {desc}")

        if action == "query_faq":
            result = search_faq(search_term)
            print(f"     {len(result)} characters retrieved from FAQ")

        elif action == "query_csv":
            result = query_csv(search_term)
            print("     CSV query completed")

        elif action == "search_web":
            result = search_web(search_term)
            print("     Web search completed")

        elif action == "register_ticket":
            data = params if params else {}
            result = register_ticket(
                customer=data.get("customer", "Customer"),
                email=data.get("email", "customer@email.com"),
                category=data.get("category", "General"),
                description=desc,
                product=data.get("product", ""),
                priority=data.get("priority", "Medium"),
            )
            print(f"     {result}")

        else:
            result = f"Unknown action: {action}"
            print(f"     {result}")

        results.append({
            "task_id": tid,
            "action": action,
            "description": desc,
            "result": result,
        })

    print(f"\n   {len(results)} result(s) collected")
    return {"execution_results": results}


print("Executor Agent defined")

## Step 5 — Agent 3: Reviewer

The Reviewer receives everything and:
1. **Reviews** whether all subtasks were addressed
2. **Identifies** gaps or missing information
3. **Generates** a final response to the customer
4. **Creates** a structured Markdown report

In [ ]:
def agent_reviewer(state: MultiAgentState) -> dict:
    """Reviewer Agent — reviews, consolidates, and generates a report."""

    print("\n" + "=" * 70)
    print("REVIEWER — Reviewing and generating report...")
    print("=" * 70)

    plan = state.get("plan", {})
    results = state.get("execution_results", [])
    question = state["question"]

    results_str = json.dumps(results, ensure_ascii=False, indent=2)
    plan_str = json.dumps(plan, ensure_ascii=False, indent=2)

    prompt = f"""You are the Reviewer Agent for TechMart's customer support center.

Original customer request:
{question}

Executed plan:
{plan_str}

Execution results:
{results_str}

Your task:
1. Review whether ALL points of the request were addressed
2. Identify gaps or missing information
3. Generate a complete final response for the customer
4. Create a report in Markdown

Return ONLY a valid JSON (no markdown) with:
{{"review": "your observations", "final_response": "response to customer", "report_md": "complete Markdown report"}}"""

    resp = llm.invoke(prompt)
    try:
        result = json.loads(resp.content)
    except json.JSONDecodeError:
        result = {
            "review": "Could not generate structured review.",
            "final_response": resp.content,
            "report_md": f"# Report\n\n{resp.content}",
        }

    print(f"   Review: {result.get('review', '')[:120]}...")
    print("   Markdown report generated!")

    return {
        "final_response": result.get("final_response", ""),
        "report_md": result.get("report_md", ""),
    }


print("Reviewer Agent defined")

## Step 6 — Building the Multi-Agent Graph

The graph is **linear** but each node is a complete agent:

```
START -> planner -> executor -> reviewer -> END
```

The simplicity of the graph is intentional: it shows that the **complexity is inside the agents**, not in the topology.

In [ ]:
graph = StateGraph(MultiAgentState)

graph.add_node("planner",  agent_planner)
graph.add_node("executor", agent_executor)
graph.add_node("reviewer", agent_reviewer)

graph.add_edge(START, "planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "reviewer")
graph.add_edge("reviewer", END)

app = graph.compile()

print("Multi-agent graph compiled!")

In [ ]:
from IPython.display import Image, display

print("Graph compiled!\n")
print("Mermaid diagram of the graph:")
display(Image(app.get_graph().draw_mermaid_png()))

## Step 7 — Test with a Complex Request

Let's test with the request that truly justifies the multi-agent architecture:

> *"I'm Maria Silva (maria.silva@email.com). My TechBook Essentials has a flickering screen. I opened a ticket before but the problem persists. I want to know: (1) the exchange policy for this case, (2) the status of my previous ticket, (3) if this model has a manufacturer recall, and (4) I want to open a new urgent ticket."*

This question requires:
- FAQ query (exchange policy)
- CSV query (previous ticket)
- Web search (manufacturer recall)
- Ticket registration (new ticket)

In [ ]:
complex_question = (
    "I'm Maria Silva (maria.silva@email.com). "
    "My TechBook Essentials has a flickering screen. "
    "I opened a ticket before but the problem persists. "
    "I want to know: (1) the exchange policy for this case, "
    "(2) the status of my previous ticket, "
    "(3) if this model has any manufacturer recall, "
    "and (4) I want to open a new urgent ticket."
)

result = app.invoke({
    "question": complex_question,
    "execution_results": [],
})

print("\n" + "=" * 70)
print("FINAL RESPONSE TO CUSTOMER:")
print("=" * 70)
print(result["final_response"])

In [ ]:
# Display the Markdown report generated by the Reviewer
print("\n" + "=" * 70)
print("GENERATED MARKDOWN REPORT:")
print("=" * 70)
print(result["report_md"])

In [ ]:
# Save the report to a file
output_path = f"data/outputs/report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result["report_md"])
print(f"\nReport saved to: {output_path}")

## Step 8 — Test with a Simple Request (for comparison)

Even a simple question goes through all 3 agents — but the plan will have fewer tasks.

In [ ]:
simple_result = app.invoke({
    "question": "Which mouse do you recommend for gaming?",
    "execution_results": [],
})

print("\n" + "=" * 70)
print("RESPONSE (simple question):")
print("=" * 70)
print(simple_result["final_response"])

## Exercises

1. **Feedback loop**: If the Reviewer identifies gaps, return to the Executor for more information (add a conditional edge between reviewer and executor).

2. **4th agent — Classifier**: Before the Planner, create an agent that classifies the urgency and sentiment of the customer.

3. **Richer report**: Have the Reviewer include in the report: processing time, sources consulted, and confidence level.

4. **Export to HTML**: Instead of Markdown, generate a formatted HTML report.

---

## Comparison Between the 3 Scenarios

| Aspect | Scenario 1 | Scenario 2 | Scenario 3 |
|---|---|---|---|
| **Agents** | 1 | 1 | 3 (Planner + Executor + Reviewer) |
| **Sources** | 2 (FAISS) | 4 (FAISS + CSV + Web) | 4 (all) |
| **Routing** | Simple conditional | Multi-step loop | Linear between agents |
| **Writing** | No | Yes (CSV) | Yes (CSV + report) |
| **Complexity** | Low | Medium | High |
| **When to use** | Simple FAQ | Operational support | Processes with planning + review |

## Key Concepts Learned

| Concept | Where It Appears |
|---|---|
| `StateGraph` | All scenarios |
| `add_node` / `add_edge` | All scenarios |
| `add_conditional_edges` | Scenarios 1 and 2 |
| `Annotated[list, add]` | Scenarios 2 and 3 |
| Dynamic routing | Scenarios 1 and 2 |
| Controlled loop | Scenario 2 |
| Agents as nodes | Scenario 3 |
| Structured output (Markdown) | Scenario 3 |